# Design check only — four responses, then stop
This notebook does **not** evaluate classification accuracy. It does not read query reference labels, run the held-out test, produce full explanations, or launch further experiments.

It checks whether Qwen can finish a decision with a short justification. A 160-character limit is enforced by the JSON generation parser. Two public validation inputs exercise agreement/conflict between the numerical ranking and a retrieved case; each is run under C and D.

**Four calls maximum.** Every attempted response is saved. Rerunning resumes only missing calls, not failed attempts. Successful formatting is not proof of correct diagnosis or evidence grounding. The four raw outputs need review afterwards.

Use your available **L4 GPU**. Run these cells in order or use **Run all**. Do not run the earlier recovery/evaluation notebook.


## 1. Connect Drive and check file sync

In [2]:
from pathlib import Path
import os, sys, json, hashlib, subprocess
from google.colab import drive
ROOT=Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
if not ROOT.is_dir() and not Path('/content/drive/MyDrive').is_dir():
    try:
        drive.mount('/content/drive')
    except ValueError as exc:
        raise RuntimeError('Drive connection failed. Reconnect the runtime and complete Google authorization. Do not mount the project folder.') from exc
if not ROOT.is_dir():
    raise FileNotFoundError(f'Project not available at {ROOT}. Check Drive sync and the Files panel.')
os.chdir(ROOT)
EXPECTED={'phm_recovery_validation.py': 'c80869cc61e66a208def269c992314cfe4cd4c5b14c9c74aab11b80eced751c3', 'phm_recovery_export.py': 'a98eed81a37334ebe501f4c4bd10571a9abc7686330587b802adddfd8f66412d', 'test_phm_recovery_validation.py': '9fd28ca6b966b3bef8ac3cf6dd94a213b0f67ac2ffde6aa6b35279224b919e0b', 'phm_diagnosis.py': 'a0b2af61d7d1a1c16512359f1e8029aae48380fc4b698c188f1f8caacf14b2fd', 'phm_two_stage.py': '07791b9b7bedc092292edfbbd34ec4ade1c1b983809860e2c8102d8672261d6c', 'phm_diagnosis_qwen.py': 'f3c9938f7fd2638de842139db5f5d2716c57f4839860291279e45c4d47e049bb', 'phm_run_reporting.py': 'fa54efc1c88773ca86e0e1a2343ff83dc34fff5a3d2545dd0a8c9e51415da2a2', 'phm_evidence_selection.py': 'bb8cd412496f3d055d11b6d0f4ac47e970f24d43b4c19493d9ff57fca8b4fac9', 'phm_output_schema.py': '01913f0c804300723c841d7c689be5105749c1e3d9c0bc110d7a9b204ea048e9', 'phm_final_metrics.py': '2ab1bba259b2bd7b4b3572d26bf0a78cb388bacdb64c548898fac4df7474c139', 'phm_heldout_export.py': '7589a9bcfdf62fc8b79cd5770e01f2b2c62bc69b0f82936756e3a43d3a7ca93d', 'phm_filtered_selection_pilot.py': '846c072def4f68fed21e04f366e5fea9c26da7f7547e238dc99bcbce3f42642e', 'phm_condition_d.py': 'e22457642e68013b1cfd4201bb6951630711d7d4890fdb28659d70dafa7a80c6', 'phm_literature_index.py': '725709141868ae494f1c515f67df5aec4451c0d137d461b3991c7ba436fb7f37', 'experiment_config.json': '563acb189f79b8f6335900a164502d01895409bee9747ee6a0f70073c63e9b3b', 'requirements-qwen-colab.txt': '0290713affb7b64776db6d09da7735ef3abbf12c38cf511c473abb0ec76f4226', 'phm_literature.py': '250f344c5a47f53e252ebe6a336270c55713f0dcf852f676e56ec6fba1e34a77', 'requirements-literature-colab.txt': '9dbbad4b5ea0136c7c6b0b4a0b0f309a62ced876b9ed9a56f662c5d593d7755c', 'requirements-literature.txt': 'f5cd09751cbd27138aa0795de219829b1fb30e48920a1dbb1df80ef1e0660a3b', 'phm_design_smoke.py': 'd24e4ac69929c079c957c2f3e1b231f732b3650eefd622af626828a99810e7c7', 'test_phm_design_smoke.py': 'cefbf34498c0233e62ad92011281cdd98e4404f20a43c859b44e92ce0b9cbdd4'}
for name, expected in EXPECTED.items():
    assert hashlib.sha256((ROOT/'Files'/name).read_bytes()).hexdigest()==expected, f'Not fully synced: {name}'
print('Code verified. No reference labels opened.')


Mounted at /content/drive
Code verified. No reference labels opened.


## 2. Install dependencies

In [3]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print('GPU:',torch.cuda.get_device_name(0))
subprocess.run([sys.executable,'-m','pip','install','-q','-r','Files/requirements-literature-colab.txt',
                '-r','Files/requirements-qwen-colab.txt','onnxruntime==1.31.1','lm-format-enforcer==0.11.3'],check=True)


GPU: Tesla T4


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', 'Files/requirements-literature-colab.txt', '-r', 'Files/requirements-qwen-colab.txt', 'onnxruntime==1.31.1', 'lm-format-enforcer==0.11.3'], returncode=0)

## 3. Verify the enforced limit and prepare four requests

In [4]:
subprocess.run([sys.executable,'-m','unittest','discover','-s','Files','-p','test_phm_design_smoke.py','-v'],check=True)
subprocess.run([sys.executable,'Files/phm_design_smoke.py','--root',str(ROOT),'--prepare-only'],check=True)


CompletedProcess(args=['/usr/bin/python3', 'Files/phm_design_smoke.py', '--root', '/content/drive/Othercomputers/My Laptop (1)/CP5403', '--prepare-only'], returncode=0)

## 4. Generate four responses and stop

In [5]:
subprocess.run([sys.executable,'-u','Files/phm_design_smoke.py','--root',str(ROOT)],check=True)
latest=json.loads((ROOT/'Outputs/LATEST_RUN.json').read_text(encoding='utf-8'))
folder=ROOT/latest['run_directory']
print('Latest report:',latest)
print('Engineering summary:',json.loads((folder/'engineering_summary.json').read_text(encoding='utf-8')))
print('Raw responses:',folder/'REVIEW_OUTPUTS.txt')
print('STOP. Let Drive sync and send the report. No accuracy evaluation or larger run follows.')


Latest report: {'run_id': '20261123T092813_four_response_design_smoke_11ad609c', 'run_directory': 'Outputs/runs/20261123T092813_four_response_design_smoke_11ad609c', 'status': 'completed', 'updated_utc': '2026-11-23T09:34:06.230419+00:00'}
Engineering summary: {'expected_responses': 4, 'valid_outputs': 4, 'token_limit_hits': 0, 'statuses': ['valid', 'valid', 'valid', 'valid'], 'next_action': 'STOP for raw-output review. No larger run is authorised by this result.', 'accuracy_evaluated': False, 'semantic_grounding_established': False}
Raw responses: /content/drive/Othercomputers/My Laptop (1)/CP5403/Outputs/runs/20261123T092813_four_response_design_smoke_11ad609c/REVIEW_OUTPUTS.txt
STOP. Let Drive sync and send the report. No accuracy evaluation or larger run follows.
